
Zillow Test

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [4]:
links = fetch_website_links("https://www.redfin.com/city/16661/MO/St-Louis")
links

['/houses-near-me',
 '/houses-near-me',
 '/condos-near-me',
 '/land-near-me',
 '/open-houses-near-me',
 '/why-buy',
 '/premier',
 '/how-much-house-can-i-afford',
 '/guides/buy',
 '/buy-a-home/classes-and-events',
 '/us-housing-market',
 '/rentals',
 '/rentals/renter-dashboard',
 '/news/rental-tracker/',
 '/how-much-rent-can-i-afford',
 '/rent-vs-buy-calculator',
 '/guides/first-time-renters-guide',
 '/rentals/list-my-home-for-rent',
 '/rentals/rental-listing-dashboard',
 '/us-rental-market',
 '/blog/should-you-sell-or-rent-your-home/',
 '/why-sell?inquirySource=484',
 '/what-is-my-home-worth',
 '/what-is-my-home-worth',
 '/why-sell?inquirySource=484',
 '/premier?inquirySource=515',
 '/sell-a-home/address?inquirySource=484',
 '/real-estate-agents?nav=sell',
 '/guides/sell/resources',
 '/sell-a-home/home-sale-proceeds-calculator',
 '/us-home-trends',
 '/mortgage-purchase-or-refinance?context=82',
 '/mortgage-get-pre-approved?context=82',
 '/todays-mortgage-rates',
 '/mortgage-rates?loanT

In [21]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
Please decide which of these are relevant web links for a purchasing a home in St-Louis,
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://www.redfin.com/MO/Saint-Louis/address/home/numbers"}
    ]
}
"""

In [22]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a purchasing a home in St-Louis, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
Please only include links that are relevant to housing information such as listing information, real estate agents, and do not include links to the company's home page, about page, or navigation pages.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [23]:
print(get_links_user_prompt("https://www.redfin.com/city/16661/MO/St-Louis"))


Here is the list of links on the website https://www.redfin.com/city/16661/MO/St-Louis -
Please decide which of these are relevant web links for a purchasing a home in St-Louis, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
Please only include links that are relevant to housing information such as listing information, real estate agents, and do not include links to the company's home page, about page, or navigation pages.

Links (some might be relative links):

/houses-near-me
/houses-near-me
/condos-near-me
/land-near-me
/open-houses-near-me
/why-buy
/premier
/how-much-house-can-i-afford
/guides/buy
/buy-a-home/classes-and-events
/us-housing-market
/rentals
/rentals/renter-dashboard
/news/rental-tracker/
/how-much-rent-can-i-afford
/rent-vs-buy-calculator
/guides/first-time-renters-guide
/rentals/list-my-home-for-rent
/rentals/rental-listing-dashboard
/us-rental-market
/blog/should-you-sell-or-rent-your-home/
/why-sell?inquiry

In [24]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [27]:
select_relevant_links("https://www.redfin.com/city/16661/MO/St-Louis")

{'links': [{'type': 'agents page',
   'url': 'https://www.redfin.com/city/16661/MO/St-Louis/real-estate/agents'},
  {'type': 'property listing',
   'url': 'https://www.redfin.com/MO/Saint-Louis/5237-Heege-Rd-63123/home/93557924'},
  {'type': 'property listing',
   'url': 'https://www.redfin.com/MO/Saint-Louis/3627-Utah-Pl-63116/home/93700109'},
  {'type': 'property listing',
   'url': 'https://www.redfin.com/MO/Saint-Louis/3877-Utah-Pl-63116/home/93728589'},
  {'type': 'property listing',
   'url': 'https://www.redfin.com/MO/Saint-Louis/3568-Queens-Hill-Dr-63129/home/93615405'},
  {'type': 'property listing',
   'url': 'https://www.redfin.com/MO/Saint-Louis/5424-Marquette-Ave-63139/home/77286062'},
  {'type': 'property listing',
   'url': 'https://www.redfin.com/MO/Saint-Louis/4695-Primm-St-63116/home/93775233'},
  {'type': 'property listing',
   'url': 'https://www.redfin.com/MO/Saint-Louis/3241-Geyer-Ave-63104/home/93689578'},
  {'type': 'property listing',
   'url': 'https://www.red